In [5]:
import os
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import warnings
from scipy.optimize import minimize_scalar
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
PROCESSED_DATA_PATH = "../processed-data"

In [ ]:
print("1. Đọc dữ liệu đã processed...")
df = pd.read_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Ép kiểu cho LightGBM / XGBoost
object_cols = [col for col in df.select_dtypes(include=['object']).columns if col not in ['Date', 'Split']]
for col in object_cols:
    df[col] = df[col].astype('category')

# Lấy tập Train gốc
train_full = df[df['Split'] == 'Train'].copy()
cols_to_drop = ['Date', 'Split', 'Revenue', 'COGS']
features = [col for col in train_full.columns if col not in cols_to_drop]

# ==========================================
# 2. THIẾT LẬP HYPERPARAMETERS (Đã chống Overfit)
# ==========================================
lgb_params = {
    'objective': 'regression', 'metric': 'mae', 
    'learning_rate': 0.02, 'num_leaves': 63, 'min_child_samples': 15,
    'subsample': 0.8, 'colsample_bytree': 0.7, 
    'reg_alpha': 0.1, 'reg_lambda': 0.5, 
    'n_estimators': 3000, 'random_state': SEED, 'n_jobs': -1, 'verbose': -1
}

xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'mae', 
    'learning_rate': 0.02, 'max_depth': 6, # Giảm max_depth
    'subsample': 0.8, 'colsample_bytree': 0.7, 
    'alpha': 0.1, 'lambda': 0.5, 
    'n_estimators': 3000, 'random_state': SEED, 'n_jobs': -1,
    'early_stopping_rounds': 50
}

# ==========================================
# 3. TIME-SERIES CROSS VALIDATION & TỐI ƯU TRỌNG SỐ
# ==========================================
print("\n--- BẮT ĐẦU TIME-SERIES CROSS VALIDATION ---")
validation_years = [2020, 2021, 2022]
results = []

# Biến lưu trữ mô hình của năm 2022 để làm SHAP
best_lgb_rev_model = None 
best_X_val = None

for val_year in validation_years:
    print(f"\n>>> Đánh giá trên năm Validation: {val_year} <<<")
    
    # Chia Train/Val theo thời gian
    train_set = train_full[train_full['year'] < val_year].copy()
    val_set = train_full[train_full['year'] == val_year].copy()
    
    X_train = train_set[features]
    X_val = val_set[features]
    
    # ÁP DỤNG LOG TRANSFORM (Khắc phục nhiễu & đỉnh nhọn)
    y_train_rev = np.log1p(train_set['Revenue'])
    y_val_rev = np.log1p(val_set['Revenue'])
    
    # --- A. HUẤN LUYỆN REVENUE ---
    model_lgb_rev = lgb.LGBMRegressor(**lgb_params)
    model_lgb_rev.fit(X_train, y_train_rev, eval_set=[(X_val, y_val_rev)], callbacks=[lgb.early_stopping(50, verbose=False)])
    
    model_xgb_rev = xgb.XGBRegressor(**xgb_params, enable_categorical=True)
    model_xgb_rev.fit(X_train, y_train_rev, eval_set=[(X_val, y_val_rev)], verbose=False)
    
    # Dự báo và chuyển ngược hàm Log (expm1)
    pred_lgb_rev = np.expm1(model_lgb_rev.predict(X_val))
    pred_xgb_rev = np.expm1(model_xgb_rev.predict(X_val))
    actual_rev = val_set['Revenue']
    
    # TÌM TRỌNG SỐ ENSEMBLE TỐI ƯU (Thay vì 50/50 cứng nhắc)
    def rev_objective(w):
        return mean_absolute_error(actual_rev, w * pred_lgb_rev + (1 - w) * pred_xgb_rev)
    
    w_rev = minimize_scalar(rev_objective, bounds=(0, 1), method='bounded').x
    final_pred_rev = w_rev * pred_lgb_rev + (1 - w_rev) * pred_xgb_rev
    print(f"Trọng số tối ưu (Revenue) -> LGBM: {w_rev:.2f} | XGB: {1 - w_rev:.2f}")
    
    # Lưu lại model 2022 để vẽ SHAP
    if val_year == 2022:
        best_lgb_rev_model = model_lgb_rev
        best_X_val = X_val
        best_w_rev_2022 = w_rev

    # --- B. SEQUENTIAL COGS (Dùng doanh thu để dự báo giá vốn) ---
    y_train_cogs = np.log1p(train_set['COGS'])
    y_val_cogs = np.log1p(val_set['COGS'])
    
    # Thêm Revenue dự báo vào features của COGS
    X_train_cogs = X_train.copy()
    X_train_cogs['predicted_revenue'] = train_set['Revenue'] # Khi train, dùng actual revenue
    
    X_val_cogs = X_val.copy()
    X_val_cogs['predicted_revenue'] = final_pred_rev # Khi test, dùng revenue vừa dự báo được!
    
    model_lgb_cogs = lgb.LGBMRegressor(**lgb_params)
    model_lgb_cogs.fit(X_train_cogs, y_train_cogs, eval_set=[(X_val_cogs, y_val_cogs)], callbacks=[lgb.early_stopping(50, verbose=False)])
    
    model_xgb_cogs = xgb.XGBRegressor(**xgb_params, enable_categorical=True)
    model_xgb_cogs.fit(X_train_cogs, y_train_cogs, eval_set=[(X_val_cogs, y_val_cogs)], verbose=False)
    
    pred_lgb_cogs = np.expm1(model_lgb_cogs.predict(X_val_cogs))
    pred_xgb_cogs = np.expm1(model_xgb_cogs.predict(X_val_cogs))
    actual_cogs = val_set['COGS']
    
    def cogs_objective(w):
        return mean_absolute_error(actual_cogs, w * pred_lgb_cogs + (1 - w) * pred_xgb_cogs)
    
    w_cogs = minimize_scalar(cogs_objective, bounds=(0, 1), method='bounded').x
    final_pred_cogs = w_cogs * pred_lgb_cogs + (1 - w_cogs) * pred_xgb_cogs
    print(f"Trọng số tối ưu (COGS)    -> LGBM: {w_cogs:.2f} | XGB: {1 - w_cogs:.2f}")
    
    # --- C. TỔNG KẾT KẾT QUẢ NĂM ---
    r2_rev = r2_score(actual_rev, final_pred_rev)
    r2_cogs = r2_score(actual_cogs, final_pred_cogs)
    print(f"Kết quả năm {val_year} -> R2_Revenue: {r2_rev:.4f} | R2_COGS: {r2_cogs:.4f}")
    results.append({'year': val_year, 'R2_Rev': r2_rev, 'R2_COGS': r2_cogs})

# Đánh giá chung
res_df = pd.DataFrame(results)
print("\n=== TỔNG KẾT CROSS-VALIDATION ===")
print(res_df)
print(f"R2 Revenue Trung bình: {res_df['R2_Rev'].mean():.4f}")
print(f"R2 COGS Trung bình:    {res_df['R2_COGS'].mean():.4f}")

# ==========================================
# 4. KHẢ NĂNG GIẢI THÍCH (SHAP ANALYSIS TRÊN NĂM 2022)
# ==========================================
print("\n--- GIAI ĐOẠN PHÂN TÍCH SHAP ---")
print("Đang tính toán SHAP values từ mô hình LightGBM (Năm 2022)...")

explainer = shap.TreeExplainer(best_lgb_rev_model)
shap_values = explainer.shap_values(best_X_val)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, best_X_val, plot_type="bar", show=False)
plt.title("Mức độ quan trọng của các đặc trưng (Năm 2022)")
plt.tight_layout()
plt.savefig('shap_importance_bar.png', dpi=300)
plt.close()

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, best_X_val, show=False)
plt.title("Tác động của đặc trưng lên Doanh thu (SHAP Beeswarm - Năm 2022)")
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=300)
plt.close()

print("SHAP Figures saved.")

1. Đọc dữ liệu đã processed...

--- BẮT ĐẦU TIME-SERIES CROSS VALIDATION ---

>>> Đánh giá trên năm Validation: 2020 <<<
Trọng số tối ưu (Revenue) -> LGBM: 0.89 | XGB: 0.11
Trọng số tối ưu (COGS)    -> LGBM: 0.06 | XGB: 0.94
Kết quả năm 2020 -> R2_Revenue: 0.7944 | R2_COGS: 0.7908

>>> Đánh giá trên năm Validation: 2021 <<<
Trọng số tối ưu (Revenue) -> LGBM: 0.62 | XGB: 0.38
Trọng số tối ưu (COGS)    -> LGBM: 0.00 | XGB: 1.00
Kết quả năm 2021 -> R2_Revenue: 0.7884 | R2_COGS: 0.7636

>>> Đánh giá trên năm Validation: 2022 <<<
Trọng số tối ưu (Revenue) -> LGBM: 0.00 | XGB: 1.00
Trọng số tối ưu (COGS)    -> LGBM: 1.00 | XGB: 0.00
Kết quả năm 2022 -> R2_Revenue: 0.7851 | R2_COGS: 0.7581

=== TỔNG KẾT CROSS-VALIDATION ===
   year    R2_Rev   R2_COGS
0  2020  0.794351  0.790767
1  2021  0.788426  0.763581
2  2022  0.785054  0.758074
R2 Revenue Trung bình: 0.7893
R2 COGS Trung bình:    0.7708

--- GIAI ĐOẠN PHÂN TÍCH SHAP ---
Đang tính toán SHAP values từ mô hình LightGBM (Năm 2022)...
SHAP F